**Title:** LRR Case Study - Part 1 (Company & People Data Modeling)<br>
**Created By:** Begimay Ahmatova<br>
**Created Date:** 2026-10-02<br>
**Description:** Notebook ingests company.csv, person.csv, and persontocompany.csv raw files into final deduplicated company and people tables.<br>

In [1]:
## libraries
import duckdb as db

In [2]:
%load_ext sql

In [3]:
con = db.connect("llr.duckdb")

In [4]:
%sql con

**Raw Ingestion**

In [5]:
%%sql
CREATE OR REPLACE TABLE raw_company AS 
SELECT * 
FROM read_csv_auto('company.csv')

Running query in 'DuckDBPyConnection'

Count
1024


In [6]:
%%sql
CREATE OR REPLACE TABLE raw_person AS
SELECT *
FROM read_csv_auto('person.csv')

Running query in 'DuckDBPyConnection'

Count
1870


In [7]:
%%sql
CREATE OR REPLACE TABLE raw_persontocompany AS
SELECT *
FROM read_csv_auto('persontocompany.csv')

Running query in 'DuckDBPyConnection'

Count
1983


**Deduping Company Data**

In [8]:
%%sql
CREATE OR REPLACE TABLE bronze_company AS
SELECT
    id,
    name,
    informalname,
    website,
    domain,
    phonenumber,
    regexp_replace(coalesce(phonenumber, ''), '[^0-9]', '', 'g') AS digits,
    city,
    state,
    postalcode,
    country,
    foundingyear,
    companytype,
    parentid,
    parentcompany,
    totalamountinvested,
    nolongerinbusiness,
    modifieddate,
    cast(modifieddate as date) as mdate
FROM raw_company

Running query in 'DuckDBPyConnection'

Count
1024


In [9]:
%%sql
CREATE OR REPLACE TABLE company_map AS
SELECT
    id as raw_id,
    first_value(id) OVER (
        PARTITION BY domain
        ORDER BY mdate DESC
    ) as final_id
FROM bronze_company

Running query in 'DuckDBPyConnection'

Count
1024


In [ ]:
%%sql
CREATE OR REPLACE TABLE deduped_company AS
WITH silver_company as (
    SELECT
        id,
        name,
        trim(regexp_replace(regexp_replace(name, '[.,\s]+$', ''),
            '(?i)[,\s]+(inc|llc|co|ltd|corp)\.?$','')) as cleaned_name,
        informalname,
        lower(trim(informalname)) as cleaned_informalname,
        website,
        domain,
        phonenumber,
        CASE
            WHEN starts_with(trim(coalesce(phonenumber, '')), '+')
                THEN '+' || digits
            WHEN length(digits) = 10
                AND lower(trim(country)) IN ('united states', 'canada')
                THEN '+1' || digits
            ELSE digits
        END AS normalized_phone,
        city,
        state,
        postalcode,
        country,
        foundingyear,
        companytype,
        parentid,
        parentcompany,
        totalamountinvested,
        nolongerinbusiness,
        modifieddate,
        cast(modifieddate as date) as modified_date
    FROM bronze_company
)
    SELECT
        s.id,
        cleaned_name as normalized_name,
        informalname,
        cleaned_informalname,
        website,
        domain,
        normalized_phone,
        city,
        state,
        postalcode,
        country,
        foundingyear,
        companytype,
        parentid,
        parentcompany,
        totalamountinvested,
        nolongerinbusiness,
        modified_date
    FROM silver_company s
    INNER JOIN company_map m ON s.id = m.raw_id AND m.raw_id = m.final_id


Running query in 'DuckDBPyConnection'

Count
999


In [11]:
%%sql
SELECT *
FROM deduped_company

Running query in 'DuckDBPyConnection'

id,normalized_name,informalname,cleaned_informalname,website,domain,normalized_phone,city,state,postalcode,country,foundingyear,companytype,parentid,parentcompany,totalamountinvested,nolongerinbusiness,modified_date
9700,Medi-Code,Medi-Code,medi-code,http://medi-code.com,medi-code.com,+18324372029,Katy,Texas,77494,United States,2009,1,None,None,None,False,2026-09-15
20074,Marketware,Marketware,marketware,https://marketware.com,marketware.com,+18019444230,Sandy,Utah,84094,United States,2014,4,1665037,Medsphere Systems Corporation,5900000,False,2026-07-16
28462,VCNetwork,VCNetwork,vcnetwork,https://vcnetwork.org,vcnetwork.org,,San Francisco,California,94114,United States,1997,5,None,None,None,False,2026-09-11
35837,Complipoint,Complipoint,complipoint,https://www.complipoint.com,complipoint.com,+18586732174,San Diego,California,92127,United States,2002,8,204227,Applications International Corp. (AIC),None,False,2025-11-26
45311,FlexiInternational Software,FlexiInternational Software,flexiinternational software,https://www.flexi.com,flexi.com,+18003539492,Shelton,Connecticut,06484,United States,1992,3,None,None,None,False,2026-08-08
48678,Blackhawk Specialty Tools,Blackhawk,blackhawk,http://www.blackhawkst.com,blackhawkst.com,+17134664200,Houston,Texas,77042,United States,2008,4,63706,"Expro International Group, Ltd.",None,False,2026-08-15
67565,DemandFarm,DemandFarm,demandfarm,https://www.demandfarm.com,demandfarm.com,+14157353143,New York City,New York,10111,United States,2016,1,None,None,None,False,2026-07-03
71013,Sutoer Solutions,Sutoer Solutions,sutoer solutions,https://sutoer.com,sutoer.com,+16303214702,Hinsdale,Illinois,60521,United States,1997,1,None,None,None,False,2026-07-07
83110,Implantech Associates,Implantech,implantech,https://www.implantech.com,implantech.com,+18053399415,Ventura,California,93003,United States,1987,1,None,None,None,False,2026-03-08
84414,Dorado Industries,Dorado Industries,dorado industries,http://www.doradoindustries.com,doradoindustries.com,+13105441316,Rancho Palos Verdes,California,90275,United States,2006,1,None,None,None,False,2025-04-04


**Data Quality Summary: Company**<br>
There were 1,024 raw company rows. After deduping, the count dropped to 999 rows. Delta of 25<br>
<br>
At this level of granularity, I checked for accidental over-merges or under-merges by manually scanning the survivorship results and comparing them to the raw data.<br>
<br>
Obviously, this method would not work at 100x scale, so I would add a flag in the table where country, year, names disagree and review only those manually.<br>
An under-merge would require fuzzy matching logic to catch two slightly similar pairs.<br>
An over-merge would require checking other fields besides name to see if there are true differences (e.g. different founding years, different locations, etc.)<br>



**Explanation/Write-Up**:<br>
<br>
I looked at multiple contenders for a field to dedupe on: name, informalname, website, domain, and phonenumber. Ultimately, I chose domain because when grouping each contendor, the results ended up being: name (1015 raw, 1010 after cleaning) < informalname (1004 raw, 1002 after cleaning) < website (1002) < domain (999) < phonenumber (902).<br>
<br>
Phonenumber technically had a lower dupe count, but it is unreliable, not only because of nulls, but also due to data quality issues, as some actual name dupes have differing numbers and some unrelated names have the same (plus an outlier +UK number that does not match the country field). Domain, on the other hand, has no nulls, cleanly resolves vastly differing names or informalnames that would not match (even after standardizing), and has the second-best dupe count.<br>
<br>
This process involved looking at the data, grouping by the fields, comparing the outputs after grouping using except, cleaning up the data, then grouping and comparing again using except, etc. Please take a look at the part1_exploratory nb for more details.<br>
<br>
In order to link a person record with a non-survivor company record, a company mapping table was needed.<br>
<br>
If I had to extend this logic 100x, then my method would potentially change based on the data. For example, if there are nulls in domain in the full dataset, then I would either A: use a multi-step matching logic (start with domain, default to informalname when domain is null, default to name when informalname is null) or B: create a fuzzy matching logic on name (fuzzy matching has a degree of ambiguity so this would be the less conservative option).<br>
<br>

**Deduping Person Data**

In [12]:
%%sql
CREATE OR REPLACE TABLE deduped_person AS (
with person as (
    select *,
        trim(firstname) as fname,
        trim(lastname) as lname,
        fname || ' ' || lname as fullname
    from raw_person
)
select *
from person
qualify row_number() over (partition by fullname, linkedin order by modifieddate desc) = 1
order by fullname asc
)

Running query in 'DuckDBPyConnection'

Count
1863


In [13]:
%%sql
select *
from deduped_person

Running query in 'DuckDBPyConnection'

id,firstname,middlename,lastname,linkedin,facebook,twitter,website,city,state,country,deleted,statecode,countrycode,modifieddate,loaddate,fname,lname,fullname
101390,Alexander,None,Brown,https://www.linkedin.com/in/alexander-brown-53,None,None,None,None,None,United States,False,None,US,2026-12-22,2026-09-01,Alexander,Brown,Alexander Brown
100002,Alexander,None,Cooper,None,None,None,None,None,None,United States,False,None,US,2026-05-14,2026-09-01,Alexander,Cooper,Alexander Cooper
101546,Alexander,None,Cruz,https://www.linkedin.com/in/alexander-cruz-10,None,None,None,None,None,United States,False,None,US,2026-08-23,2026-09-01,Alexander,Cruz,Alexander Cruz
100195,Alexander,None,Gonzalez,None,None,None,None,None,None,United States,False,None,US,2026-04-22,2026-09-01,Alexander,Gonzalez,Alexander Gonzalez
100251,Alexander,None,Gutierrez,https://www.linkedin.com/in/alexander-gutierrez-60,None,None,None,None,None,United States,False,None,US,2024-07-28,2026-09-01,Alexander,Gutierrez,Alexander Gutierrez
101454,Alexander,None,Hall,https://www.linkedin.com/in/alexander-hall-96,None,None,None,None,None,United States,False,None,US,2026-04-05,2026-09-01,Alexander,Hall,Alexander Hall
100188,Alexander,None,Hill,https://www.linkedin.com/in/alexander-hill-85,None,None,None,None,None,United States,False,None,US,2024-06-22,2026-09-01,Alexander,Hill,Alexander Hill
101712,Alexander,None,Kelly,None,None,None,None,None,None,United States,False,None,US,2026-05-27,2026-09-01,Alexander,Kelly,Alexander Kelly
100077,Alexander,None,King,https://www.linkedin.com/in/alexander-king-91,None,None,None,None,None,United States,False,None,US,2026-01-30,2026-09-01,Alexander,King,Alexander King
101630,Alexander,None,King,https://www.linkedin.com/in/alexander-king-37,None,None,None,None,None,United States,False,None,US,2025-10-29,2026-09-01,Alexander,King,Alexander King


**Data Quality Summary: Person**<br>
Raw count was 1,870. Final count ended up being 1,863. Delta of 7.<br>

**Explanation/Write-Up:**<br>
Since the same person can have differing ids, I had to dedupe person data.<br> 
<br>
Unlike company data, people data doesn't have a lot of options to dedupe on.<br>
My contendors were name or linkedin. Linkedin had more nulls so I first chose to dedupe on name, which had a higher delta, but the problem was that different linkedin values existed for the same name. So I chose to use linkedin as the second qualifier.<br>
Since firstname and lastname were split into seperate fields, I merged them to get the fullname of each person and chose the latest record based on modifieddate.<br>



**Deduping PersonToCompany Data**

In [14]:
%%sql
CREATE OR REPLACE TABLE deduped_ptc AS (
    select *
    from raw_persontocompany
    qualify row_number() over (partition by personid order by modifieddate desc) = 1
)

Running query in 'DuckDBPyConnection'

Count
1870


In [15]:
%%sql
select *
from deduped_ptc

Running query in 'DuckDBPyConnection'

id,personid,companyid,startdate,enddate,email,emailconfirmed,phone,title,role,deleted,modifieddate,loaddate
500001,100001,9700,2017-08-10,None,None,False,None,Chief Executive Officer,1,False,2024-02-24,2026-09-01
500002,100002,9700,2024-01-29,None,alexander.cooper@medi-code.com,True,(425) 659-5557,Chief Technology Officer,2,False,2024-11-22,2026-09-01
500003,100003,20074,2024-07-03,None,None,False,None,Chief Executive Officer,1,False,2025-06-25,2026-09-01
500004,100004,28462,2018-02-12,None,jonathan.thomas@vcnetwork.org,True,(438) 303-7227,Chief Executive Officer,1,False,2024-11-29,2026-09-01
500005,100005,28462,2021-10-31,None,stephen.nguyen@vcnetwork.org,True,(899) 863-2169,Chief Financial Officer,2,False,2025-05-16,2026-09-01
500006,100006,28462,2022-09-05,None,pamela.harris@vcnetwork.org,True,(904) 770-4598,Director of Sales,3,False,2024-04-24,2026-09-01
500007,100007,45311,2017-12-20,2025-05-10,andrew.rodriguez@flexi.com,True,(858) 669-3340,Chief Executive Officer,1,False,2025-06-22,2026-09-01
500008,100008,48678,2019-04-26,None,justin.phillips@blackhawkst.com,True,(632) 810-2040,Chief Executive Officer,1,False,2026-12-19,2026-09-01
500009,100009,48678,2023-01-31,2026-01-04,amy.lopez@blackhawkst.com,True,None,Managing Partner,1,False,2025-11-27,2026-09-01
500010,100010,67565,2023-07-21,2025-12-22,ryan.smith@demandfarm.com,True,(403) 356-7126,Chief Executive Officer,1,False,2025-10-25,2026-09-01


In [16]:
%%sql
with dupes as (
    select personid
    from raw_persontocompany
    group by personid
    having count(*) > 1
)
select count(*) from (
select
    r.personid,
    r.id,
    r.companyid,
    r.startdate,
    r.enddate,
    r.modifieddate,
    (r.id = s.id) as is_survivor
from raw_persontocompany r
join deduped_ptc s on r.personid = s.personid
where r.personid in (select personid from dupes)
  and s.enddate is not null
order by r.personid, r.modifieddate desc
) 
where is_survivor = True and enddate is not null


Running query in 'DuckDBPyConnection'

count_star()
21


**Data Quality Summary: PersonToCompany**<br>
Raw count was 1,983. Final count ended up being 1,870. Delta of 113.<br>

**Explanation/Write-Up**:<br>
<br>
Since persontocompany is a mapping table and we need to use it to get a personid matched with a companyid, we need to dedupe this table and get 1 record per personid.<br>
Therefore, I chose to dedupe on personid with the latest modifieddate.<br>
<br>
Since we assume that the surviving record is the most up to date based on the modified date (typically when a record is modified, it is to update previously false information or to add previously missing information), the data quality concern is when 2 records with the same personid have differing startdates and enddates (where the older record does not have an enddate but the newer record does). We end up having 522 records with a non-null enddate, with 21 records potentially showing a person that may have left when they still work there or 19 records showing a person that supposedly does work there but may have left.<br>


In [17]:
%%sql
CREATE OR REPLACE TABLE final_people AS (
select p.*,
    m.final_id as company_id,
    c.normalized_name,
    ptc.startdate,
    ptc.enddate,
    (ptc.enddate is null) as is_current
from deduped_person p
left join deduped_ptc ptc on p.id = ptc.personid
left join company_map m on ptc.companyid = m.raw_id
left join deduped_company c on m.final_id = c.id
)

Running query in 'DuckDBPyConnection'

Count
1863


**Explanation/Write-Up**:<br>
For the final person table, we carry through the relationship to company_map table in order to show where all person records work(ed) at, even when we have dropped the dupe company record in favor of its survivor.<br>
<br>
Additionally, the assumption is that deal teams are only concerned with employees that have current ties to their companies, so surviving records with an enddate would not be helpful to them, in which case I added an is_current flag to help them filter out out of scope employees.<br>

In [ ]:
%%sql
select *
from final_people

Running query in 'DuckDBPyConnection'

id,firstname,middlename,lastname,linkedin,facebook,twitter,website,city,state,country,deleted,statecode,countrycode,modifieddate,loaddate,fname,lname,fullname,company_id,normalized_name,startdate,enddate,is_current
101390,Alexander,None,Brown,https://www.linkedin.com/in/alexander-brown-53,None,None,None,None,None,United States,False,None,US,2026-12-22,2026-09-01,Alexander,Brown,Alexander Brown,9619605,MAD Technical Solutions,2017-04-11,2018-01-11,False
100002,Alexander,None,Cooper,None,None,None,None,None,None,United States,False,None,US,2026-05-14,2026-09-01,Alexander,Cooper,Alexander Cooper,9700,Medi-Code,2024-01-29,None,True
101546,Alexander,None,Cruz,https://www.linkedin.com/in/alexander-cruz-10,None,None,None,None,None,United States,False,None,US,2026-08-23,2026-09-01,Alexander,Cruz,Alexander Cruz,15173053,Elite Mergers & Acquisitions,2015-06-15,None,True
100195,Alexander,None,Gonzalez,None,None,None,None,None,None,United States,False,None,US,2026-04-22,2026-09-01,Alexander,Gonzalez,Alexander Gonzalez,443586,LeedSource Group,2020-06-27,2023-01-05,False
100251,Alexander,None,Gutierrez,https://www.linkedin.com/in/alexander-gutierrez-60,None,None,None,None,None,United States,False,None,US,2024-07-28,2026-09-01,Alexander,Gutierrez,Alexander Gutierrez,544524,Die Trying TX,2015-09-19,None,True
101454,Alexander,None,Hall,https://www.linkedin.com/in/alexander-hall-96,None,None,None,None,None,United States,False,None,US,2026-04-05,2026-09-01,Alexander,Hall,Alexander Hall,12188544,HighKey Agency,2022-03-27,2024-01-03,False
100188,Alexander,None,Hill,https://www.linkedin.com/in/alexander-hill-85,None,None,None,None,None,United States,False,None,US,2024-06-22,2026-09-01,Alexander,Hill,Alexander Hill,440496,Northern Plains Track,2016-08-02,None,True
101712,Alexander,None,Kelly,None,None,None,None,None,None,United States,False,None,US,2026-05-27,2026-09-01,Alexander,Kelly,Alexander Kelly,16292129,Allied Employer Services,2024-07-28,None,True
100077,Alexander,None,King,https://www.linkedin.com/in/alexander-king-91,None,None,None,None,None,United States,False,None,US,2026-01-30,2026-09-01,Alexander,King,Alexander King,216941,Boston MedFlight (BMF),2021-01-10,2026-02-02,False
101630,Alexander,None,King,https://www.linkedin.com/in/alexander-king-37,None,None,None,None,None,United States,False,None,US,2025-10-29,2026-09-01,Alexander,King,Alexander King,15818021,Dyskami Publishing,2021-09-09,2022-01-04,False


**Risks & Monitoring:**<br>
<br>
- Company, phone: unreliable (nulls, different numbers for the same company, the same number on unrelated companies, a UK outlier). Monitor: track the share of null phones and any number shared across different domains, as a list of possible missed duplicates.
- PersonToCompany, conflicting dates: modifieddate and start and end dates disagree, so the surviving link may not be the true one. Monitor: count people whose duplicate rows disagree on enddate after each sync, and send them to the source owner.
- Person, same name with different LinkedIn: treated as different people, but it could be a LinkedIn data error. Monitor: count same-name groups with conflicting or blank LinkedIn and review a sample.

<br>
DQ Checks I'd add:<br>
<br>
1. Company id and domain are unique and not null in the final companies table.<br>
2. Every raw company id maps to a company in the final table (no link left pointing to a merged-away company).<br>
3. Person id is unique, and every person's company exists in the final companies table.<br>
4. No merged company pair disagrees on country (over-merge signal).